# DSCI 552 Homework 3

**Name:** Jingyu Zhang  
**GitHub Username:** jingyuz9-wq  
**USC ID:** 5744431585

In [1]:
from pathlib import Path
from io import StringIO

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import bootstrap

In [3]:
DATA_DIR = Path("../data/AReM")

print("Data folder exists:", DATA_DIR.exists())
print("Activity folders:", sorted(
    folder.name for folder in DATA_DIR.iterdir() if folder.is_dir()
))

Data folder exists: True
Activity folders: ['bending1', 'bending2', 'cycling', 'lying', 'sitting', 'standing', 'walking']


## 1. Time Series Classification, Part 1

### 1(a) AReM Data Set

The AReM data set contains seven types of human activities and 88 instances. Each instance contains six time series, and each time series contains 480 consecutive measurements.

Source: [UCI AReM Data Set](https://archive.ics.uci.edu/dataset/366/activity+recognition+system+based+on+multisensor+data+fusion+arem)

In [5]:
COLUMN_NAMES = [
    "time",
    "avg_rss12",
    "var_rss12",
    "avg_rss13",
    "var_rss13",
    "avg_rss23",
    "var_rss23"
]

def read_sensor_csv(file_path):
    lines = Path(file_path).read_text(encoding="utf-8").splitlines()

    data_lines = [
        line.strip().rstrip(",")
        for line in lines
        if line.strip() and not line.lstrip().startswith("#")
    ]

    separator = "," if "," in data_lines[0] else r"\s+"

    df = pd.read_csv(
        StringIO("\n".join(data_lines)),
        sep=separator,
        header=None,
        names=COLUMN_NAMES,
        engine="python"
    )

    return df.apply(pd.to_numeric, errors="raise")


ACTIVITIES = [
    "bending1",
    "bending2",
    "cycling",
    "lying",
    "sitting",
    "standing",
    "walking"
]

In [6]:
test_file = DATA_DIR / "bending1" / "dataset1.csv"

df_test = read_sensor_csv(test_file)

print(df_test.shape)
df_test.head()

(480, 7)


,time,avg_rss12,var_rss12,avg_rss13,var_rss13,avg_rss23,var_rss23
0,0,39.25,0.43,22.75,0.43,33.75,1.3
1,250,39.25,0.43,23.00,0.00,33.00,0.0
2,500,39.25,0.43,23.25,0.43,33.00,0.0
3,750,39.50,0.50,23.00,0.71,33.00,0.0
4,1000,39.50,0.50,24.00,0.00,33.00,0.0


In [10]:
def get_split(activity, dataset_num):
    if activity in ["bending1", "bending2"]:
        return "test" if dataset_num in [1, 2] else "train"
    else:
        return "test" if dataset_num in [1, 2, 3] else "train"

In [12]:
train_files = []
test_files = []

for activity in ACTIVITIES:
    folder = DATA_DIR / activity

    for file_path in sorted(folder.glob("dataset*.csv")):
        dataset_num = int(file_path.stem.replace("dataset", ""))

        if get_split(activity, dataset_num) == "test":
            test_files.append(file_path)
        else:
            train_files.append(file_path)

print("Training instances:", len(train_files))
print("Testing instances:", len(test_files))
print("Total instances:", len(train_files) + len(test_files))

Training instances: 69
Testing instances: 19
Total instances: 88


### 1(c)i. Time-Domain Features

Common time-domain features used in time series classification include:

- Minimum
- Maximum
- Mean
- Median
- Standard deviation
- Variance
- First quartile (Q1)
- Third quartile (Q3)
- Interquartile range (IQR)
- Range
- Root mean square (RMS)
- Skewness
- Kurtosis
- Mean absolute deviation
- Zero-crossing rate

These features describe the statistical characteristics of time series data, including central tendency, variability, and distribution.

### 1(c)ii. Extract Time-Domain Features

Extract the minimum, maximum, mean, median, standard deviation, first quartile, and third quartile from each of the six time series.

In [13]:
series = df_test["avg_rss12"]

print("Minimum:", series.min())
print("Maximum:", series.max())
print("Mean:", series.mean())
print("Median:", series.median())
print("Standard Deviation:", series.std())
print("First Quartile:", series.quantile(0.25))
print("Third Quartile:", series.quantile(0.75))

Minimum: 37.25
Maximum: 45.0
Mean: 40.62479166666667
Median: 40.5
Standard Deviation: 1.4769669162668115
First Quartile: 39.25
Third Quartile: 42.0


In [14]:
FEATURE_COLUMNS = COLUMN_NAMES[1:]

def extract_features(file_path):
    df = read_sensor_csv(file_path)
    features = {}

    for i, column in enumerate(FEATURE_COLUMNS, start=1):
        series = df[column]

        features[f"min{i}"] = series.min()
        features[f"max{i}"] = series.max()
        features[f"mean{i}"] = series.mean()
        features[f"median{i}"] = series.median()
        features[f"std{i}"] = series.std()
        features[f"1st_quart{i}"] = series.quantile(0.25)
        features[f"3rd_quart{i}"] = series.quantile(0.75)

    return features

In [16]:
all_files = train_files + test_files

feature_rows = []

for i, file_path in enumerate(all_files):
    print(i + 1, file_path, flush=True)

    features = extract_features(file_path)
    feature_rows.append(features)

feature_df = pd.DataFrame(feature_rows)

print(feature_df.shape)
feature_df.head()

1 ../data/AReM/bending1/dataset3.csv
2 ../data/AReM/bending1/dataset4.csv
3 ../data/AReM/bending1/dataset5.csv
4 ../data/AReM/bending1/dataset6.csv
5 ../data/AReM/bending1/dataset7.csv
6 ../data/AReM/bending2/dataset3.csv
7 ../data/AReM/bending2/dataset4.csv
8 ../data/AReM/bending2/dataset5.csv
9 ../data/AReM/bending2/dataset6.csv
10 ../data/AReM/cycling/dataset10.csv
11 ../data/AReM/cycling/dataset11.csv
12 ../data/AReM/cycling/dataset12.csv
13 ../data/AReM/cycling/dataset13.csv
14 ../data/AReM/cycling/dataset14.csv
15 ../data/AReM/cycling/dataset15.csv
16 ../data/AReM/cycling/dataset4.csv
17 ../data/AReM/cycling/dataset5.csv
18 ../data/AReM/cycling/dataset6.csv
19 ../data/AReM/cycling/dataset7.csv
20 ../data/AReM/cycling/dataset8.csv
21 ../data/AReM/cycling/dataset9.csv
22 ../data/AReM/lying/dataset10.csv
23 ../data/AReM/lying/dataset11.csv
24 ../data/AReM/lying/dataset12.csv
25 ../data/AReM/lying/dataset13.csv
26 ../data/AReM/lying/dataset14.csv
27 ../data/AReM/lying/dataset15.csv
2

,min1,max1,mean1,median1,std1,1st_quart1,3rd_quart1,min2,max2,mean2,...,std5,1st_quart5,3rd_quart5,min6,max6,mean6,median6,std6,1st_quart6,3rd_quart6
0,35.00,47.40,43.954500,44.33,1.558835,43.00,45.00,0.0,1.70,0.426250,...,1.999604,35.3625,36.50,0.0,1.79,0.493292,0.43,0.513506,0.00,0.94
1,33.00,47.75,42.179812,43.50,3.670666,39.15,45.00,0.0,3.00,0.696042,...,3.849448,30.4575,36.33,0.0,2.18,0.613521,0.50,0.524317,0.00,1.00
2,33.00,45.75,41.678063,41.75,2.243490,41.33,42.75,0.0,2.83,0.535979,...,2.411026,28.4575,31.25,0.0,1.79,0.383292,0.43,0.389164,0.00,0.50
3,37.00,48.00,43.454958,43.25,1.386098,42.50,45.00,0.0,1.58,0.378083,...,2.488862,22.2500,24.00,0.0,5.26,0.679646,0.50,0.622534,0.43,0.87
4,36.25,48.00,43.969125,44.50,1.618364,43.31,44.67,0.0,1.50,0.413125,...,3.318301,20.5000,23.75,0.0,2.96,0.555312,0.49,0.487826,0.00,0.83


### 1(c)iii. Bootstrap Confidence Intervals

Estimate the standard deviation of each time-domain feature and construct 90% bootstrap confidence intervals.

In [17]:
feature_std = feature_df.std()

print(feature_std)

min1          9.569975
max1          4.394362
mean1         5.335718
median1       5.440054
std1          1.772153
1st_quart1    6.153590
3rd_quart1    5.138925
min2          0.000000
max2          5.062729
mean2         1.574164
median2       1.412244
std2          0.884105
1st_quart2    0.946386
3rd_quart2    2.125266
min3          2.956462
max3          4.875137
mean3         4.008380
median3       4.036396
std3          0.946710
1st_quart3    4.220658
3rd_quart3    4.171628
min4          0.000000
max4          2.183625
mean4         1.166114
median4       1.145586
std4          0.458242
1st_quart4    0.843620
3rd_quart4    1.552504
min5          6.124001
max5          5.741238
mean5         5.675593
median5       5.813782
std5          1.024898
1st_quart5    6.096465
3rd_quart5    5.531720
min6          0.045838
max6          2.518921
mean6         1.154812
median6       1.086474
std6          0.517617
1st_quart6    0.758584
3rd_quart6    1.523599
dtype: float64


In [18]:
def std_statistic(data):
    return np.std(data, ddof=1)

bootstrap_results = []

for column in feature_df.columns:
    values = feature_df[column].to_numpy()

    result = bootstrap(
        (values,),
        std_statistic,
        confidence_level=0.90,
        n_resamples=1000,
        method="percentile",
        random_state=42
    )

    bootstrap_results.append({
        "Feature": column,
        "Standard Deviation": feature_std[column],
        "CI Lower": result.confidence_interval.low,
        "CI Upper": result.confidence_interval.high
    })

bootstrap_df = pd.DataFrame(bootstrap_results)

bootstrap_df

,Feature,Standard Deviation,CI Lower,CI Upper
0,min1,9.569975,8.275241,10.747623
1,max1,4.394362,3.343823,5.296316
2,mean1,5.335718,4.686470,5.869648
3,median1,5.440054,4.788491,6.016169
4,std1,1.772153,1.569967,1.945903
5,1st_quart1,6.153590,5.547790,6.646684
6,3rd_quart1,5.138925,4.330721,5.825098
7,min2,0.000000,0.000000,0.000000
8,max2,5.062729,4.645742,5.403547
9,mean2,1.574164,1.411542,1.710131


### 1(c)iv. Important Time-Domain Features

In my opinion, the three most important time-domain features are mean, standard deviation, and maximum.

- **Mean:** It represents the average level of sensor measurements and helps distinguish activities with different overall movement patterns.

- **Standard Deviation:** It measures the variability of sensor readings and helps differentiate activities with stable movements from those with more dynamic movements.

- **Maximum:** It captures the peak values of sensor measurements, which may help identify activities involving sudden or intense movements.

These three features provide useful information about the central tendency, variability, and extreme values of time series data.

## 2. ISLR 3.7.4

### (a) Linear Relationship — Training RSS

The cubic regression is expected to have a lower or equal training RSS than the linear regression.

This is because the cubic model is more flexible and includes the linear model as a special case. Adding polynomial terms cannot increase the training RSS when using least squares.

### (b) Linear Relationship — Test RSS

The linear regression is expected to have a lower test RSS than the cubic regression.

Since the true relationship is linear, the cubic model introduces unnecessary complexity and may overfit the training data. Therefore, the simpler linear model is expected to generalize better to unseen data.

### (c) Non-linear Relationship — Training RSS

The cubic regression is expected to have a lower or equal training RSS than the linear regression.

This is because the cubic model is more flexible and includes the linear model as a special case. Therefore, its training RSS cannot be higher than that of the linear model, even when the true relationship is non-linear.

### (d) Non-linear Relationship — Test RSS

It is difficult to tell which model will have a lower test RSS.

If the relationship is highly non-linear, the cubic model might fit better. However, if the relationship is only slightly non-linear, the linear model could perform better because it is simpler and less likely to overfit.

Therefore, we cannot determine which model has a lower test RSS without more information.